# Week 3: Exploratory Data Analysis & Feature Engineering
## Notebook 10: Feature Interactions: Polynomials & Cartesian Feature Crosses

### 1. Conceptual Intuition & Theoretical Background
Linear models assume features contribute additively to target predictions. However, in physical, financial, and behavioral systems, the effect of feature $X_1$ often depends strongly on the value of feature $X_2$ (**Interaction Effect**).

**1. Multiplicative Feature Interactions:**
$$y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 (X_1 \cdot X_2) + \epsilon$$
The marginal effect of $X_1$ becomes dynamic:
$$\frac{\partial y}{\partial X_1} = \beta_1 + \beta_3 X_2$$

**2. Cartesian Categorical Feature Crosses:**
Combining two discrete categorical features into a joint composite feature:
$$\text{Device} \in \{\text{iOS}, \text{Android}\} \times \text{Browser} \in \{\text{Safari}, \text{Chrome}\} \longrightarrow \text{Device\_Browser} \in \{\text{iOS\_Safari}, \dots\}$$
Enables linear models to learn non-linear boundary configurations without deep neural network architectures.

---

### 2. Scratch Python Implementation: Interaction Transformer

In [ ]:
import numpy as np
import pandas as pd

class InteractionFeatureExtractorScratch:
    def __init__(self, include_bias=False):
        self.include_bias = include_bias
        
    def fit_transform(self, X_df):
        cols = list(X_df.columns)
        n = len(cols)
        out_df = X_df.copy()
        
        # Compute 2nd-degree multiplicative interactions: X_i * X_j
        for i in range(n):
            for j in range(i, n):
                col_name = f"{cols[i]}_x_{cols[j]}"
                out_df[col_name] = X_df[cols[i]] * X_df[cols[j]]
                
        if self.include_bias:
            out_df.insert(0, 'bias', 1.0)
            
        return out_df

# Demonstration: Synthetic Pricing Dynamics
# Revenue depends non-linearly on Price * Conversion_Rate
np.random.seed(42)
df_market = pd.DataFrame({
    'price': np.random.uniform(10.0, 100.0, 5),
    'ctr': np.random.uniform(0.01, 0.08, 5),
    'impressions': np.random.randint(1000, 5000, 5)
})

transformer = InteractionFeatureExtractorScratch()
poly_features = transformer.fit_transform(df_market)

# Categorical Feature Cross Demo
df_users = pd.DataFrame({
    'country': ['US', 'UK', 'US', 'DE'],
    'subscription': ['Free', 'Premium', 'Premium', 'Free']
})
df_users['country_x_sub'] = df_users['country'] + "_" + df_users['subscription']

print("--- Original Market Features ---")
print(df_market)
print("\n--- Generated Polynomial & Cross Features ---")
print(poly_features.columns.tolist())
print("\n--- Categorical Feature Crosses ---")
print(df_users[['country', 'subscription', 'country_x_sub']])

### 3. Step-by-Step Code Walkthrough
1. **Combinatorial Pairing:** Loops over unique feature pairs $(i, j)$ to generate pairwise product interactions.
2. **Feature Cross Encoding:** Concatenates string categories into distinct joint interaction buckets.
3. **Linear Expressiveness:** Allows linear algorithms to learn non-linear boundaries.

---

### 4. Industry Applications & Trade-offs
- **Ad Click-Through Rate (CTR) Prediction (Google DLRM / Wide & Deep):** Crossing `[User_Topic, Ad_Category]` powers candidate ranking engines.
- **Trade-off:** High feature dimensionality; crossing $D$ features yields $O(D^2)$ columns, risking overfitting without regularization.

---

### 5. Top Interview Questions & Answers
**Q1: How do tree-based models handle feature interactions compared to linear models?**
> **Answer:** Decision trees learn interactions implicitly through sequential hierarchical splits (e.g. split on $X_1$, then split on $X_2$). Linear models require interactions to be engineered explicitly.

**Q2: What is the risk of generating high-degree polynomial features ($d \ge 3$)?**
> **Answer:** Severe multicollinearity, exponential dimensionality explosion, and numerical instability when raising continuous values to high powers.